# 第五阶段：智能指针

## 实验 3：唯一所有权的转移

`std::unique_ptr<T>` 表示同一时刻最多只有一个 owner。实验 2 已观察到它会自动清理资源；本实验进一步研究 ownership 如何沿变量和函数边界转移。

本实验关注：

- 为什么唯一所有权不能复制；
- `std::move` 前后源 owner 与目标 owner 的状态；
- 参数类型如何区分只读借用、可空借用和消费 ownership；
- ownership 如何从工厂经过调用方转移到长期持有者；
- 为什么移动后的对象仍然有效，但不再拥有原资源。

右值引用和移动构造函数的实现细节留到下一阶段；这里先学会从 API 类型读取所有权契约。

In [1]:
// 本步骤：引入智能指针、移动、类型检查和断言所需的标准库。
#include <cassert>
#include <iostream>
#include <memory>
#include <type_traits>
#include <utility>

### 1. 准备可观察的资源类型

下面的 `User` 记录构造、析构和当前存活数量。移动 `unique_ptr` 时不应该重新构造或复制 `User`，只应改变哪个智能指针负责最终销毁它。

In [2]:
// 本步骤：定义带身份和生命周期日志的 User，用于追踪同一个对象。
class User
{
public:
    explicit User(int id)
        : id_(id)
    {
        // 构造开始一段 User 生命周期，并增加存活计数。
        ++live_count_;
        std::cout << "User " << id_ << " created, live = " << live_count_ << '\n';
    }

    ~User() noexcept
    {
        // 析构结束生命周期；唯一所有权应确保这一步只发生一次。
        --live_count_;
        std::cout << "User " << id_ << " destroyed, live = " << live_count_ << '\n';
    }

    int id() const noexcept
    {
        // 返回对象身份的值副本，不转移任何 ownership。
        return id_;
    }

    static int live_count() noexcept
    {
        // 返回当前存活数，供每段实验验证清理结果。
        return live_count_;
    }

private:
    int id_;
    inline static int live_count_ = 0;
};

### 2. 复制在编译期被禁止

以下代码不能编译：

```cpp
auto first = std::make_unique<User>(1);
auto second = first; // error: unique_ptr 的复制构造函数已删除
```

如果复制成功，`first` 和 `second` 都会声称自己是唯一 owner，并在析构时对同一地址执行两次 `delete`。因此标准库直接删除复制构造和复制赋值，让错误停在编译期。它与前一阶段不可复制的 `File` RAII wrapper 是同一原则。

In [ ]:
// 本步骤：用类型特征验证 unique_ptr 不可复制、但支持移动。
static_assert(!std::is_copy_constructible_v<std::unique_ptr<User>>);
static_assert(!std::is_copy_assignable_v<std::unique_ptr<User>>);
static_assert(std::is_move_constructible_v<std::unique_ptr<User>>);
static_assert(std::is_move_assignable_v<std::unique_ptr<User>>);

### 3. 移动把 ownership 交给新 owner

`std::move(first)` 本身只是把 `first` 标记为可被移动；随后 `unique_ptr` 的移动构造函数取走其中的地址。被管理的 `User` 没有重新构造，也没有改变地址。

移动后，源 `unique_ptr` 仍是一个可以检查、赋值和析构的有效对象，但它的规定状态必为空。

In [3]:
// 本步骤：移动构造新 owner，观察对象地址、存活数和源指针状态。
{
    const int before = User::live_count();

    {
        auto first = std::make_unique<User>(1);

        // 记录非拥有地址，只用于证明移动前后仍是同一个 User。
        User *address_before_move = first.get();

        // 移动构造将 ownership 从 first 转移给 second。
        std::unique_ptr<User> second = std::move(first);

        // first 不再拥有对象；second 拥有原地址，对象数量没有增加。
        assert(!first);
        assert(second.get() == address_before_move);
        assert(User::live_count() == before + 1);
        std::cout << std::boolalpha
                  << "first owns object: " << static_cast<bool>(first) << '\n'
                  << "second owns object: " << static_cast<bool>(second) << '\n';
    }

    // 只有 second 析构时释放一次 User。
    assert(User::live_count() == before);
}

User 1 created, live = 1
first owns object: false
second owns object: true
User 1 destroyed, live = 0


ownership 的变化应读成：

```text
Before                              After
first -- uniquely owns --> User     first --> empty
                                    second -- uniquely owns --> User

std::move(first) 只允许转移；unique_ptr 的移动构造完成转移。
```

### 4. 三种参数签名表达三种 contract

接口是否取得 ownership，应由参数类型直接说明：

| 参数 | 语义 | 是否允许为空 | 调用后 owner |
| --- | --- | --- | --- |
| `const User&` | 同步只读借用 | 否 | 不变 |
| `const User*` | 同步只读、可空借用 | 是 | 不变 |
| `std::unique_ptr<User>` | 接收并消费唯一 ownership | 是 | 调用方移出后变空 |

借用参数不得保存到 owner 生命周期之外；按值接收 `unique_ptr` 的函数则必须决定何时销毁或继续转移对象。

In [4]:
// 本步骤：定义必须存在的只读借用接口，不接管 User。
void inspect(const User &user)
{
    // 引用只在本次调用期间读取对象。
    std::cout << "inspect User " << user.id() << '\n';
}

// 本步骤：定义可空的只读借用接口，仍然不接管 User。
void maybe_inspect(const User *user)
{
    // pointer 可能为空，因此访问前必须检查。
    if (user == nullptr)
    {
        std::cout << "no User to inspect\n";
        return;
    }

    // 非空时只读取对象，不执行 delete。
    inspect(*user);
}

// 本步骤：定义 ownership sink；按值参数表示函数取得唯一 ownership。
void consume(std::unique_ptr<User> user)
{
    // user 是当前 owner；函数可使用对象，并在返回时自动销毁它。
    if (user)
    {
        std::cout << "consume User " << user->id() << '\n';
    }
}

In [5]:
// 本步骤：调用三种接口，观察借用不会清空 owner，消费则必须显式移动。
{
    const int before = User::live_count();
    auto owner = std::make_unique<User>(2);

    // 解引用产生必须存在的借用；owner 保持不变。
    inspect(*owner);
    assert(owner);

    // get() 产生可空借用；被调用方不会取得 ownership。
    maybe_inspect(owner.get());
    maybe_inspect(nullptr);
    assert(owner);

    // consume(owner) 无法编译；std::move 明确授权函数取走 ownership。
    consume(std::move(owner));

    // consume 返回时参数析构，User 已销毁，调用方 owner 保持为空。
    assert(!owner);
    assert(User::live_count() == before);
}

User 2 created, live = 1
inspect User 2
inspect User 2
no User to inspect
consume User 2
User 2 destroyed, live = 0


### 5. 移动后的 owner 可以重新赋值

moved-from `unique_ptr` 的状态是明确的空状态，不是悬空 pointer。可以安全检查、`reset()`，也可以让它接管另一个对象；只是不能在未判空时解引用。

In [6]:
// 本步骤：消费原对象后复用空 owner，证明移动不结束 unique_ptr 自身生命周期。
{
    const int before = User::live_count();
    auto owner = std::make_unique<User>(3);

    // ownership 被 consume 取走并在函数返回时释放。
    consume(std::move(owner));
    assert(!owner);

    // 对空的 moved-from owner 重新赋值，开始管理新的 User。
    owner = std::make_unique<User>(4);
    assert(owner && owner->id() == 4);
    assert(User::live_count() == before + 1);

    // reset 结束新 ownership，计数回到实验开始前。
    owner.reset();
    assert(User::live_count() == before);
}

User 3 created, live = 1
consume User 3
User 3 destroyed, live = 0
User 4 created, live = 1
User 4 destroyed, live = 0


### 6. ownership 可以沿多层 API 继续传递

工厂返回 `unique_ptr`，表示创建者把 ownership 交给调用方；持有者的构造函数按值接收 `unique_ptr`，表示它将接管 ownership。每个转移点都能从类型和 `std::move` 看见。

In [7]:
// 本步骤：定义工厂函数，把新 User 的 ownership 返回给调用方。
std::unique_ptr<User> create_user(int id)
{
    // 返回值携带唯一 ownership，不暴露 owning raw pointer。
    return std::make_unique<User>(id);
}

// 本步骤：定义长期持有者，通过构造参数接管 User ownership。
class UserSession
{
public:
    explicit UserSession(std::unique_ptr<User> user)
        : user_(std::move(user))
    {
        // 参数先取得 ownership，再移动到成员中延长管理生命周期。
        assert(user_);
        std::cout << "session owns User " << user_->id() << '\n';
    }

    const User &user() const noexcept
    {
        // 返回只读借用；引用不能超过 UserSession 的生命周期。
        return *user_;
    }

private:
    std::unique_ptr<User> user_;
};

In [8]:
// 本步骤：追踪 ownership 从工厂、局部变量到 UserSession 的完整路径。
{
    const int before = User::live_count();

    {
        // 工厂返回 ownership，pending 成为第一个调用方 owner。
        auto pending = create_user(5);

        // 构造 session 时显式转移 ownership，pending 随后变空。
        UserSession session(std::move(pending));
        assert(!pending);

        // getter 只提供借用，ownership 仍在 session 的成员中。
        inspect(session.user());
        assert(User::live_count() == before + 1);
    }

    // session 析构带动成员 unique_ptr 析构，最终释放 User。
    assert(User::live_count() == before);
}

User 5 created, live = 1
session owns User 5
inspect User 5
User 5 destroyed, live = 0


### 7. API 设计与 ABI 边界

看到函数签名时，应直接翻译为：

```cpp
void inspect(const User &user);
// 借用只读 User；必须存在，不拿走 ownership

void maybe_inspect(const User *user);
// 借用只读 User；可能为空，不拿走 ownership

void consume(std::unique_ptr<User> user);
// 接收唯一 ownership；调用方必须显式移动

std::unique_ptr<User> create_user();
// 返回唯一 ownership；调用方负责继续管理
```

`std::unique_ptr` 依赖 C++ 模板、析构和标准库 ABI，所以这些签名只适合作为 C++ 内部 API。跨 C ABI 时应使用 opaque handle，并提供 `create/destroy` 函数；Kotlin/Native 包装层再把 handle 封装成可关闭的 owner。

## 实验结论

唯一所有权的核心不是“指针只能有一个变量”，而是：**同一时刻只有一个对象负责最终释放资源。** 可以存在多个临时 raw pointer/reference borrower，但只有一个 owner。

```text
create_user()
    │ returns ownership
    ▼
pending
    │ std::move
    ▼
UserSession::user_
    │ session 析构
    ▼
~User()
```

需要记住：

1. 复制 ownership 会产生歧义，因此 `unique_ptr` 禁止复制；
2. `std::move` 表示允许转移，具体转移由接收类型的移动操作完成；
3. 移动后的 `unique_ptr` 有效但为空，可以检查和重新赋值；
4. 按值接收 `unique_ptr` 是明确的 ownership sink；
5. 只使用对象时传引用或 pointer，不要无故转移 ownership。